If you're using IDE like VS Code, please install the **Jupyter** extension first.

The repeated failures (`Py4JNetworkError`) were caused by **insufficient memory allocated to Docker Desktop**, which made the Spark driver crash when processing large files.

**Please follow these steps exactly:**
1.  **Adjust Docker memory**:
    *   Locate the Docker whale icon in your system tray (bottom-right on Windows).
    *   Right-click it -> **Settings**.
    *   Left menu -> **Resources**.
    *   Find the **Memory** slider and **set it to at least 10 GB** (12 GB recommended if your machine has 16 GB or more).
    *   Click **Apply & Restart**.
     ### Or
    * WIN + R -> %USERPROFILE%
    * create a file named .wslconfig
    * add the following line to the file:

In [ ]:
    [wsl2]
    memory=12GB

2.  **Restart the notebook kernel**:
    *   After Docker has restarted, return to this page. 
    *   Follow the hint in Phase 1&2 to restart the kernel.
3.  **Run the cells below in order**.
4.  **Please don't run the second cell only if you have to**.

**if you are fluently aborted from the JVM, please follow the step above**

## [README]
This notebook require docker desktop.

once downloaded, type the following command in the terminal:

In [ ]:
docker-compose up -d

Once the command is executed, type the following command in the terminal to get the token:

## Alternative
Run the following command to generate a token, then paste it into the link below.

## Recommended
**copy the output link** and replace the number after the http:// with localhost.


In [ ]:
docker exec jupyter_analytics jupyter server list

Find the token in the terminal outputs, it would look like this:

**http://localhost:8888/lab?token=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX**

Note: the ip address might be different

Copy that token link. Then, go to your notebook and click **Select Kernel** in the top-right corner.

Go to **Select Another Kernel...** > **Existing Jupyter Server...** and paste your token in the input box.

Hit Enter, then Enter again... and you're in!




Please run the first cell to initialize the spark,

If you make a mistake in editing the following cells, no worries — just switch the second cell from Markdown to Python,

run it, and switch it back.

The following cells are just AI generated and serve as a runnable skeleton.[Don't foget to remove this sentence!]

The actual implementation details and business logic need to be carefully filled in by each student.





# PHASE 3 Data Storage and Cleansing

In [1]:
import time
time_start = time.time()

In [2]:
# Must run after restarting
# Spark Session Init (8GB RAM)
# ==========================================
import os
from pyspark.sql import SparkSession

# 1. Set HDFS Identity to 'root' for full permissions
os.environ['HADOOP_USER_NAME'] = 'root'
os.chdir('/home/jovyan/work')

# 2. Stop any existing session
if SparkSession.getActiveSession():
    SparkSession.getActiveSession().stop()

# 3. Initialize Spark with 8GB Memory
spark = SparkSession.builder \
    .appName("Phase3_Process_Storage") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .config("spark.sql.caseSensitive", "true") \
    .getOrCreate()

print(f"✅ Spark Ready! Driver Memory: 8g")

✅ Spark Ready! Driver Memory: 8g


In [3]:
#Do NOT run this cell only when you crush the data or testing

#Press Enter and change markdown to python when you decide to run this cell

print("🧹 Wiping HDFS /ecommerce...")
try:
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
    fs.delete(spark._jvm.org.apache.hadoop.fs.Path("/ecommerce"), True)
    print("✅ HDFS Cleaned.")
except Exception as e:
    print(f"❌ Cleanup failed: {e}")

🧹 Wiping HDFS /ecommerce...
✅ HDFS Cleaned.


In [ ]:
# Filter null price in advance
# ==========================================
with open("datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

datasets = [
    (f"data_lake/{types}_{line.split(',')[0].strip()}.parquet", f"/ecommerce/data_lake/{types}/{types}_{line.split(',')[1].strip()}")
    for types in ['meta', 'review']
    for line in lines
]



In [4]:
# ==========================================
# Sync Local Data to HDFS (with MORE Partitions)
# ==========================================
# Ensure data have extracted to data_lake folder

with open("datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

datasets = [
    (f"data_lake/{types}_{line.split(',')[0].strip()}.parquet", f"/ecommerce/data_lake/{types}/{types}_{line.split(',')[1].strip()}")
    for types in ['meta', 'review']
    for line in lines
]

for local_p, hdfs_p in datasets:
    if os.path.exists(local_p):
        print(f"🚚 Syncing {local_p}...")
        # Repartition MORE to manage memory usage for large files
        # This splits the data into 32 smaller chunks before writing
        df = spark.read.parquet(f"file://{os.path.abspath(local_p)}").repartition(32)
        df.write.mode("overwrite").parquet(hdfs_p)
        print(f"   ✅ Done: {hdfs_p}")
    else:
        print(f"   ❌ Local file not found: {local_p}")

print("\n✨ Data synchronization complete!")

🚚 Syncing data_lake/meta_Electronics.parquet...
   ✅ Done: /ecommerce/data_lake/meta/meta_elec
🚚 Syncing data_lake/meta_Cell_Phones_and_Accessories.parquet...
   ✅ Done: /ecommerce/data_lake/meta/meta_cellp
🚚 Syncing data_lake/review_Electronics.parquet...
   ✅ Done: /ecommerce/data_lake/review/review_elec
🚚 Syncing data_lake/review_Cell_Phones_and_Accessories.parquet...
   ✅ Done: /ecommerce/data_lake/review/review_cellp

✨ Data synchronization complete!


In [5]:
meta_path = "/ecommerce/data_lake/meta/*"
review_path = "/ecommerce/data_lake/review/*"

def preview_parquet(label, path):
    print(f"\n=== {label} ===")
    print(f"Path: {path}")
    df = spark.read.parquet(path)
    print(f"Columns ({len(df.columns)}): {df.columns}")
    df.printSchema()
    df.limit(5).show(5, truncate=60)
    try:
        print(f"Partitions: {df.rdd.getNumPartitions()}")
    except Exception:
        pass

preview_parquet("RAW META (HDFS)", meta_path)
preview_parquet("RAW REVIEW (HDFS)", review_path)

print("\n✅ Data preview finished (no full count performed).")


=== RAW META (HDFS) ===
Path: /ecommerce/data_lake/meta/*
Columns (16): ['author', 'average_rating', 'bought_together', 'categories', 'description', 'details', 'features', 'images', 'main_category', 'parent_asin', 'price', 'rating_number', 'store', 'subtitle', 'title', 'videos']
root
 |-- author: struct (nullable = true)
 |    |-- about: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- avatar: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- bought_together: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- details: struct (nullable = true)
 |    |-- AC Adapter Current: string (nullable = true)
 |    |-- Accessory Connection Type: string (nullable = true)
 |    |-- Action: string (nullable = true)
 |    |-- Active Ingredient

In [6]:
# ==========================================
# Data Preview
# ==========================================
import os

DATA_LAKE = "/home/jovyan/work/data_lake"

with open("datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

files = {
    f"{types}_{line.split(',')[0].strip()}" : f"{types}_{line.split(',')[0].strip()}.parquet"
    for types in ["meta", "review"]
    for line in lines
}

# "meta_Electronics": "meta_Electronics_clean.parquet",
#     "meta_Cell_Phones": "meta_Cell_Phones_and_Accessories_clean.parquet",
#     "review_Electronics": "review_Electronics_clean.parquet",
#     "review_Cell_Phones": "review_Cell_Phones_and_Accessories_clean.parquet",

def preview_local_parquet(name, path):
    if not os.path.exists(path):
        print(f"⚠️  [{name}] not found: {path}")
        return

    size_mb = os.path.getsize(path) / (1024 * 1024)
    df = spark.read.parquet(f"file://{path}")

    print(f"\n{'='*60}")
    print(f"📋 {name}")
    print(f"{'='*60}")
    print(f"   Size    : {size_mb:.1f} MB")
    print(f"   Columns ({len(df.columns)}): {df.columns}")
    df.printSchema()
    df.limit(3).show(3, truncate=60)
    try:
        print(f"   Partitions: {df.rdd.getNumPartitions()}")
    except Exception:
        pass

for name, filename in files.items():
    preview_local_parquet(name, os.path.join(DATA_LAKE, filename))


📋 meta_Electronics
   Size    : 141.5 MB
   Columns (16): ['author', 'average_rating', 'bought_together', 'categories', 'description', 'details', 'features', 'images', 'main_category', 'parent_asin', 'price', 'rating_number', 'store', 'subtitle', 'title', 'videos']
root
 |-- author: struct (nullable = true)
 |    |-- about: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- avatar: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- bought_together: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- details: struct (nullable = true)
 |    |-- 63 99: string (nullable = true)
 |    |-- 74 89: string (nullable = true)
 |    |-- 89 89: string (nullable = true)
 |    |-- ABPA Partslink Number: string (nullable = true)
 |    |-- AC Adap

## Member 2
### Data Quality Engineer

In [7]:
# Example Data Cleaning
# ==========================================
# Note: We do NOT deduplicate by `asin`.
# Rationale: `parent_asin` is treated as the primary product-family identifier, while `asin` can represent different variants/parameters under the same parent.
# Keeping multiple rows per `parent_asin` preserves variant-level metadata for downstream analysis.

from pyspark.sql import functions as F

meta_raw = spark.read.parquet("/ecommerce/data_lake/meta/*")
review_raw = spark.read.parquet("/ecommerce/data_lake/review/*")

meta_clean = meta_raw
# if "asin" not in meta_clean.columns and "parent_asin" in meta_clean.columns:
#     meta_clean = meta_clean.withColumn("asin", F.col("parent_asin").cast("string"))
# if "title" in meta_clean.columns:
#     meta_clean = meta_clean.withColumn("title", F.trim(F.col("title")))
if "parent_asin" in meta_clean.columns:
    meta_clean = meta_clean.filter(F.col("parent_asin").isNotNull())
# elif "asin" in meta_clean.columns:
#     meta_clean = meta_clean.filter(F.col("asin").isNotNull())
if "price" in meta_clean.columns:
    price_num = F.regexp_replace(F.col("price").cast("string"), "[^0-9.]", "").cast("double")
    meta_clean = meta_clean.withColumn("price", price_num)
    #meta_clean = meta_clean.filter(F.col("price").isNotNull() & (F.col("price") > 0))
if "parent_asin" in meta_clean.columns and "asin" in meta_clean.columns:
    meta_clean = meta_clean.dropDuplicates(["parent_asin", "asin"])
elif "parent_asin" in meta_clean.columns:
    meta_clean = meta_clean.dropDuplicates(["parent_asin"])
else:
    meta_clean = meta_clean.dropDuplicates()

review_clean = review_raw

def to_unix_seconds(col):
    ts = col.cast("long")
    return F.when(ts > F.lit(20000000000), (ts / 1000).cast("long")).otherwise(ts)

if "overall" not in review_clean.columns and "rating" in review_clean.columns:
    review_clean = review_clean.withColumn("overall", F.col("rating").cast("double"))
if "reviewText" not in review_clean.columns and "text" in review_clean.columns:
    review_clean = review_clean.withColumn("reviewText", F.col("text").cast("string"))
if "reviewerID" not in review_clean.columns and "user_id" in review_clean.columns:
    review_clean = review_clean.withColumn("reviewerID", F.col("user_id").cast("string"))

if "unixReviewTime" not in review_clean.columns:
    if "timestamp" in review_clean.columns:
        review_clean = review_clean.withColumn("unixReviewTime", to_unix_seconds(F.col("timestamp")))
    elif "time" in review_clean.columns:
        review_clean = review_clean.withColumn("unixReviewTime", to_unix_seconds(F.col("time")))

if "overall" in review_clean.columns:
    review_clean = review_clean.filter((F.col("overall") >= 1) & (F.col("overall") <= 5))
if "reviewText" in review_clean.columns:
    review_clean = review_clean.withColumn("reviewText", F.trim(F.col("reviewText")))
if "asin" in review_clean.columns:
    review_clean = review_clean.filter(F.col("asin").isNotNull())
if "unixReviewTime" in review_clean.columns:
    review_clean = review_clean.withColumn("unixReviewTime", to_unix_seconds(F.col("unixReviewTime")))
    review_clean = review_clean.filter(F.col("unixReviewTime").isNotNull())
    review_clean = review_clean.withColumn("review_date", F.from_unixtime(F.col("unixReviewTime")).cast("date"))
    review_clean = review_clean.withColumn("year_month", F.date_format(F.col("review_date"), "yyyy-MM"))
dedup_keys = [c for c in ["reviewerID", "asin", "unixReviewTime"] if c in review_clean.columns]
review_clean = review_clean.dropDuplicates(dedup_keys) if dedup_keys else review_clean.dropDuplicates()

meta_out = "/ecommerce/data_lake/clean/meta"
review_out = "/ecommerce/data_lake/clean/review"

meta_partitions = max(32, spark.sparkContext.defaultParallelism)
review_partitions = max(128, spark.sparkContext.defaultParallelism * 2)

meta_clean.repartition(meta_partitions).write.mode("overwrite").parquet(meta_out)
if "year_month" in review_clean.columns:
    review_clean.repartition(review_partitions, "year_month").write.mode("overwrite").partitionBy("year_month").parquet(review_out)
else:
    review_clean.repartition(review_partitions).write.mode("overwrite").parquet(review_out)

print("✅ Full cleaning + saving finished")
print(meta_out)
print(review_out)

# Data Cleansing output
# ==========================================
from pyspark.sql import functions as F

def _cols(df, wanted):
    return [c for c in wanted if c in df.columns]

def _print_step(title, before_count, after_count):
    dropped = before_count - after_count
    pct = (dropped / before_count * 100.0) if before_count else 0.0
    print(f"\n[{title}]")
    print(f"  before: {before_count:,}")
    print(f"  after : {after_count:,}")
    print(f"  drop  : {dropped:,} ({pct:.2f}%)")

def _show_bad(df, cond, cols, n=5, label=None):
    bad = df.filter(~cond)
    sel = _cols(bad, cols)
    if label:
        print(f"\n  bad sample ({label})")
    else:
        print("\n  bad sample")
    if sel:
        bad.select(sel).limit(n).show(n, truncate=60)
    else:
        bad.limit(n).show(n, truncate=60)

print("\n==== CLEANING AUDIT (RAW -> CLEAN) ====")

meta_raw_a = meta_raw
review_raw_a = review_raw

print("\n-- RAW COUNTS --")
meta_raw_cnt = meta_raw_a.count()
review_raw_cnt = review_raw_a.count()
print(f"RAW meta   : {meta_raw_cnt:,}")
print(f"RAW review : {review_raw_cnt:,}")

print("\n-- META CLEANING --")
meta_a = meta_raw_a
if "parent_asin" not in meta_a.columns and "asin" in meta_a.columns:
    meta_a = meta_a.withColumn("parent_asin", F.col("asin").cast("string"))
if "asin" not in meta_a.columns and "parent_asin" in meta_a.columns:
    meta_a = meta_a.withColumn("asin", F.col("parent_asin").cast("string"))

c0 = meta_a.count()
if "parent_asin" in meta_a.columns:
    cond = F.col("parent_asin").isNotNull()
    _show_bad(meta_a, cond, ["parent_asin", "asin", "title", "brand", "price"], label="parent_asin is null")
    meta_a = meta_a.filter(cond)
elif "asin" in meta_a.columns:
    cond = F.col("asin").isNotNull()
    _show_bad(meta_a, cond, ["asin", "title", "brand", "price"], label="asin is null")
    meta_a = meta_a.filter(cond)
c1 = meta_a.count()
_print_step("META: filter id not null", c0, c1)

if "price" in meta_a.columns:
    price_num = F.regexp_replace(F.col("price").cast("string"), "[^0-9.]", "").cast("double")
    meta_with_price = meta_a.withColumn("_price_num", price_num)
    cond = F.col("_price_num").isNotNull() & (F.col("_price_num") > 0)
    _show_bad(meta_with_price, cond, ["parent_asin", "asin", "price", "_price_num", "title"], label="invalid or missing price")
    valid_cnt = meta_with_price.filter(cond).count()
    invalid_cnt = meta_with_price.filter(~cond).count()
    print(f"\n  price valid rows   : {valid_cnt:,}")
    print(f"  price invalid rows : {invalid_cnt:,}")
    c2b = meta_a.count()
    # Thinking for long... Should we drop invalid price rows?
    #meta_a = meta_with_price.filter(cond).drop("price").withColumnRenamed("_price_num", "price")
    c2a = meta_a.count()
    _print_step("META: normalize+filter price", c2b, c2a)

if "parent_asin" in meta_a.columns and "asin" in meta_a.columns:
    dup = meta_a.groupBy("parent_asin", "asin").count().filter(F.col("count") > 1).orderBy(F.col("count").desc())
    if dup.take(1):
        print("\n  duplicate (parent_asin, asin) sample")
        dup.limit(5).show(5, truncate=60)
    c3b = meta_a.count()
    meta_a = meta_a.dropDuplicates(["parent_asin", "asin"])
    c3a = meta_a.count()
    _print_step("META: dropDuplicates(parent_asin, asin)", c3b, c3a)
elif "parent_asin" in meta_a.columns:
    dup = meta_a.groupBy("parent_asin").count().filter(F.col("count") > 1).orderBy(F.col("count").desc())
    if dup.take(1):
        print("\n  duplicate parent_asin sample")
        dup.limit(5).show(5, truncate=60)
    c3b = meta_a.count()
    meta_a = meta_a.dropDuplicates(["parent_asin"])
    c3a = meta_a.count()
    _print_step("META: dropDuplicates(parent_asin)", c3b, c3a)

print("\n-- REVIEW CLEANING --")
review_a = review_raw_a

def to_unix_seconds(col):
    ts = col.cast("long")
    return F.when(ts > F.lit(20000000000), (ts / 1000).cast("long")).otherwise(ts)

if "overall" not in review_a.columns and "rating" in review_a.columns:
    review_a = review_a.withColumn("overall", F.col("rating").cast("double"))
if "reviewText" not in review_a.columns and "text" in review_a.columns:
    review_a = review_a.withColumn("reviewText", F.col("text").cast("string"))
if "reviewerID" not in review_a.columns and "user_id" in review_a.columns:
    review_a = review_a.withColumn("reviewerID", F.col("user_id").cast("string"))
if "unixReviewTime" not in review_a.columns:
    if "timestamp" in review_a.columns:
        review_a = review_a.withColumn("unixReviewTime", to_unix_seconds(F.col("timestamp")))
    elif "time" in review_a.columns:
        review_a = review_a.withColumn("unixReviewTime", to_unix_seconds(F.col("time")))

r0 = review_a.count()
if "overall" in review_a.columns:
    cond = (F.col("overall") >= 1) & (F.col("overall") <= 5) & F.col("overall").isNotNull()
    _show_bad(review_a, cond, ["asin", "parent_asin", "rating", "overall", "user_id", "reviewerID"], label="overall out of range")
    review_a = review_a.filter(cond)
r1 = review_a.count()
_print_step("REVIEW: filter overall 1..5", r0, r1)

if "asin" in review_a.columns:
    r2b = review_a.count()
    cond = F.col("asin").isNotNull()
    _show_bad(review_a, cond, ["asin", "parent_asin", "reviewerID", "user_id", "overall"], label="asin is null")
    review_a = review_a.filter(cond)
    r2a = review_a.count()
    _print_step("REVIEW: filter asin not null", r2b, r2a)

if "unixReviewTime" in review_a.columns:
    r3b = review_a.count()
    cond = F.col("unixReviewTime").isNotNull()
    _show_bad(review_a, cond, ["asin", "reviewerID", "timestamp", "unixReviewTime"], label="unixReviewTime is null")
    review_a = review_a.filter(cond)
    r3a = review_a.count()
    _print_step("REVIEW: filter unixReviewTime not null", r3b, r3a)

if "reviewText" in review_a.columns:
    review_a = review_a.withColumn("reviewText", F.trim(F.col("reviewText")))
    r4b = review_a.count()
    cond = (F.length(F.col("reviewText")) > 0) | F.col("reviewText").isNull()
    _show_bad(review_a, cond, ["asin", "reviewerID", "reviewText"], label="empty reviewText")
    review_a = review_a.filter(F.col("reviewText").isNotNull() & (F.length(F.col("reviewText")) > 0))
    r4a = review_a.count()
    _print_step("REVIEW: filter non-empty reviewText", r4b, r4a)

if "unixReviewTime" in review_a.columns:
    review_a = review_a.withColumn("review_date", F.from_unixtime(F.col("unixReviewTime")).cast("date"))
    review_a = review_a.withColumn("year_month", F.date_format(F.col("review_date"), "yyyy-MM"))

dedup_keys = [c for c in ["reviewerID", "asin", "unixReviewTime"] if c in review_a.columns]
if dedup_keys:
    dup = review_a.groupBy(*dedup_keys).count().filter(F.col("count") > 1).orderBy(F.col("count").desc())
    if dup.take(1):
        print("\n  duplicate key sample")
        dup.limit(5).show(5, truncate=60)
    r5b = review_a.count()
    review_a = review_a.dropDuplicates(dedup_keys)
    r5a = review_a.count()
    _print_step(f"REVIEW: dropDuplicates({dedup_keys})", r5b, r5a)

print("\n-- FINAL (AUDIT) COUNTS --")
print(f"META  final (audit): {meta_a.count():,}")
print(f"REVIEW final (audit): {review_a.count():,}")

✅ Full cleaning + saving finished
/ecommerce/data_lake/clean/meta
/ecommerce/data_lake/clean/review

==== CLEANING AUDIT (RAW -> CLEAN) ====

-- RAW COUNTS --
RAW meta   : 146,229
RAW review : 6,945,481

-- META CLEANING --

  bad sample (parent_asin is null)
+-----------+----+-----+-----+
|parent_asin|asin|title|price|
+-----------+----+-----+-----+
+-----------+----+-----+-----+


[META: filter id not null]
  before: 146,229
  after : 146,229
  drop  : 0 (0.00%)

  bad sample (invalid or missing price)
+-----------+----------+-----+----------+------------------------------------------------------------+
|parent_asin|      asin|price|_price_num|                                                       title|
+-----------+----------+-----+----------+------------------------------------------------------------+
| B000067R8G|B000067R8G| NULL|      NULL| Sony DSCP9 Cyber-shot 4MP Digital Camera w/ 3x Optical Zoom|
| B08RZFK143|B08RZFK143| NULL|      NULL|MSI GeForce RTX 3090 SUPRIM X 24G Gra

In [8]:
# Cleaned Data Summary (HDFS)
# ==========================================
meta_written = spark.read.parquet(meta_out)
review_written = spark.read.parquet(review_out)

m_count = meta_written.count()
m_columns = meta_written.columns
r_count = review_written.count()
r_columns = review_written.columns
print(meta_written.show(5))
print(review_written.show(5))

print("📊 Cleaned HDFS Data Summary:")
print(f"   - Meta Clean Records  : {m_count:,}")
print(f"   - Meta Clean Columns  : {m_columns}\n")
print(f"   - Review Clean Records: {r_count:,}")
print(f"   - Review Clean Columns: {r_columns}\n")

if "year_month" in review_written.columns:
    month_count = review_written.select("year_month").distinct().count()
    bounds = review_written.selectExpr("min(year_month) as min_ym", "max(year_month) as max_ym").collect()[0]
    print(f"   - Months             : {month_count:,} ({bounds['min_ym']} ~ {bounds['max_ym']})")

+------+--------------+---------------+----------+-----------+--------------------+--------+------+--------------------+-----------+-----+-------------+-------+--------+--------------------+------+
|author|average_rating|bought_together|categories|description|             details|features|images|       main_category|parent_asin|price|rating_number|  store|subtitle|               title|videos|
+------+--------------+---------------+----------+-----------+--------------------+--------+------+--------------------+-----------+-----+-------------+-------+--------+--------------------+------+
|  NULL|           5.0|           NULL|      NULL|       NULL|{NULL, NULL, NULL...|    NULL|  NULL|      Camera & Photo| B07VMDHFKX| NULL|            3|  Canon|    NULL|Canon RF 85mm F1....|  NULL|
|  NULL|           3.8|           NULL|      NULL|       NULL|{NULL, NULL, NULL...|    NULL|  NULL|      Camera & Photo| B07BKV8TFS| NULL|            2|   Sony|    NULL|Sony Alpha a6300 ...|  NULL|
|  NULL|  

In [9]:
# Example Data Testing
# ==========================================

from pyspark.sql import functions as F

Path = spark._jvm.org.apache.hadoop.fs.Path
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())

assert fs.exists(Path(meta_out)), f"Missing HDFS path: {meta_out}"
assert fs.exists(Path(review_out)), f"Missing HDFS path: {review_out}"

meta_written = spark.read.parquet(meta_out)
review_written = spark.read.parquet(review_out)

assert len(meta_written.take(1)) > 0, "meta_out is empty"
assert len(review_written.take(1)) > 0, "review_out is empty"

if "parent_asin" in meta_written.columns:
    assert meta_written.filter(F.col("parent_asin").isNull()).limit(1).count() == 0, "meta_clean has null parent_asin"
if "price" in meta_written.columns:
    pass
    
    # assert meta_written.filter(F.col("price").isNull()).limit(1).count() == 0, "meta_clean has null price"

if "overall" in review_written.columns:
    assert review_written.filter((F.col("overall") < 1) | (F.col("overall") > 5) | F.col("overall").isNull()).limit(1).count() == 0, "review_clean has invalid overall"
if "reviewText" in review_written.columns:
    assert review_written.filter(F.col("reviewText").isNotNull() & (F.col("reviewText") != F.trim(F.col("reviewText")))).limit(1).count() == 0, "review_clean has untrimmed reviewText"

print("✅ Full-pipeline cleaning/saving tests passed")

✅ Full-pipeline cleaning/saving tests passed


In [10]:
meta_out = "/ecommerce/data_lake/clean/meta"
review_out = "/ecommerce/data_lake/clean/review"

In [11]:
# Add Boxplot to Remove Outliers

In [12]:
# Category-aware Dual-Feature Clustering (Word2Vec + Custom Stopwords)
# ==========================================
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import CountVectorizer, Tokenizer, StopWordsRemover, Word2Vec, VectorAssembler, Normalizer

from datetime import date, datetime
from openpyxl import Workbook
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

TARGET_CATEGORIES = {
    "All Electronics": "El",
    "Camera & Photo": "Ca",
    "Cell Phones & Accessories": "Ce",
    "Computers": "Co",
    "Home Audio & Theater": "Ha",
}

CLUSTER_K = {
    "All Electronics": 10,
    "Camera & Photo": 8,
    "Cell Phones & Accessories": 10,
    "Computers": 12,
    "Home Audio & Theater": 8,
}

CUSTOM_BRANDS = [
     # Smartphones / Tablets
    "apple", "samsung", "google", "sony", "oneplus", "xiaomi", "oppo", "huawei",
    "motorola", "nokia", "realme", "vivo", "zte", "blackberry", "honor",
    # Laptops / PCs
    "dell", "hp", "lenovo", "asus", "acer", "microsoft", "razer", "msi",
    "toshiba", "gigabyte", "lg", "alienware", "system76",
    # Audio
    "bose", "jbl", "sennheiser", "jabra", "anker", "soundcore",
    "beats", "shure", "audio-technica", "skullcandy",
    "plantronics", "beyerdynamic", "marshall",
    # Gaming Peripherals
    "logitech", "corsair", "steelseries", "hyperx", "roccat",
    "zowie", "glorious", "ducky",
    # Cameras
    "canon", "nikon", "fujifilm", "gopro",
    "panasonic", "olympus", "leica", "dji", "insta360", "sigma", "pentax",
    # Displays / TVs
    "tcl", "hisense", "vizio", "philips",
    "viewsonic", "benq", "aoc",
]

print("Loading cleaned metadata...")
meta_clean = spark.read.parquet(meta_out)

def parse_details(df):
    details_dt = df.schema["details"].dataType
    if isinstance(details_dt, T.StringType):
        details_json = F.regexp_replace(F.col("details"), r"\bNone\b", "null")
        details_json = F.regexp_replace(details_json, "'", '"')
        details_map  = F.from_json(details_json, T.MapType(T.StringType(), T.StringType()))
    else:
        details_map = F.from_json(F.to_json(F.col("details")), T.MapType(T.StringType(), T.StringType()))
    return df.withColumn("detail_keys", F.map_keys(details_map))

meta_cluster_frames = []

for cate in TARGET_CATEGORIES:
    abbv_cate = TARGET_CATEGORIES[cate]
    k = CLUSTER_K[cate]

    print(f"\nProcessing Category: {cate} (Target K={k})")
    df_cat = meta_clean.filter(F.col("main_category") == cate)
    df_cat = parse_details(df_cat)
    
    # Filter valid details
    df_cat = df_cat.withColumn(
        "detail_keys",
        F.expr("filter(detail_keys, x -> x is not null and length(trim(x)) > 0)")
    ).filter(F.size(F.col("detail_keys")) > 0)

    # Filter out empty titles
    df_cat = df_cat.filter(F.col("title").isNotNull() & (F.length(F.trim(F.col("title"))) > 0))

    if df_cat.limit(1).count() == 0:
        print(f"   Skip: No valid data for {cate}")
        continue

    for col_drop in ["cv_features", "title_words", "filtered_words", "title_features", "combined_features", "norm_features", "cluster"]:
        if col_drop in df_cat.columns:
            df_cat = df_cat.drop(col_drop)

    # ---------------------------------------------------------
    # Feature Pipeline 1: Word2Vec + Brand Killer (NLP)
    # ---------------------------------------------------------
    df_cat = df_cat.withColumn("clean_title", F.lower(F.col("title")))
    # Retain only letters
    df_cat = df_cat.withColumn("clean_title", F.regexp_replace("clean_title", "[^a-z\\s]", ""))
    
    tokenizer = Tokenizer(inputCol="clean_title", outputCol="title_words")
    df_cat = tokenizer.transform(df_cat)
    
    # Core Magic: Inject Custom Brands into StopWords
    remover = StopWordsRemover(inputCol="title_words", outputCol="filtered_words")
    default_stopwords = remover.getStopWords()
    remover.setStopWords(default_stopwords + CUSTOM_BRANDS)
    df_cat = remover.transform(df_cat)
    
    # Safety Check: Remove rows where filtered_words became empty (e.g. title was just "Apple")
    df_cat = df_cat.filter(F.size(F.col("filtered_words")) > 0)
    
    # Word2Vec: Learn Semantic Embeddings (100 dimensions)
    word2Vec = Word2Vec(vectorSize=100, minCount=2, inputCol="filtered_words", outputCol="title_features", seed=42)
    w2v_model = word2Vec.fit(df_cat)
    df_cat = w2v_model.transform(df_cat)

    # ---------------------------------------------------------
    # Feature Pipeline 2: Details Structure (Keys)
    # ---------------------------------------------------------
    cv = CountVectorizer(inputCol="detail_keys", outputCol="cv_features", vocabSize=300, minDF=3.0, binary=True)
    cv_model = cv.fit(df_cat)
    df_cat = cv_model.transform(df_cat)

    # ---------------------------------------------------------
    # Feature Fusion & Normalization
    # ---------------------------------------------------------
    assembler = VectorAssembler(
        inputCols=["title_features", "cv_features"],
        outputCol="combined_features"
    )
    featurized = assembler.transform(df_cat)
    
    # NORMALIZER: Essential for combining Word2Vec (dense, small scale) and CV (sparse, int scale)
    normalizer = Normalizer(inputCol="combined_features", outputCol="norm_features", p=2.0)
    normalized_data = normalizer.transform(featurized)

    # ---------------------------------------------------------
    # KMeans Clustering
    # ---------------------------------------------------------
    km = KMeans(k=k, seed=42, featuresCol="norm_features", predictionCol="cluster", maxIter=20)
    km_model = km.fit(normalized_data)
    clustered = km_model.transform(normalized_data)

    clustered = clustered.withColumn(
        "cluster",
        F.concat(F.lit(f"{abbv_cate}_"), F.col("cluster").cast("string"))
    ).drop("cv_features", "title_features", "combined_features", "norm_features", "filtered_words", "title_words", "clean_title")

    meta_cluster_frames.append(clustered)

if not meta_cluster_frames:
    raise ValueError("No rows matched TARGET_CATEGORIES with valid details.")

print("\nUnioning all clustered categories...")
meta_cluster_all = meta_cluster_frames[0]
for df_i in meta_cluster_frames[1:]:
    meta_cluster_all = meta_cluster_all.unionByName(df_i, allowMissingColumns=True)

meta_cluster_out = "ecommerce/data_lake/cluster/meta"
meta_cluster_all.write.mode("overwrite").partitionBy("cluster").parquet(meta_cluster_out)
print('Successfully wrote to: ', meta_cluster_out)

Loading cleaned metadata...

Processing Category: All Electronics (Target K=10)


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
# Prepare data for Export to Excel
# ==========================================
meta_cluster_xlsx = "/home/jovyan/work/meta_cluster_w2v.xlsx"
max_str_len = 5000

def _sanitize_cell(v):
    if v is None:
        return None
    if isinstance(v, str):
        return ILLEGAL_CHARACTERS_RE.sub("", v)[:max_str_len]
    if isinstance(v, (int, float, bool, date, datetime)):
        return v
    return ILLEGAL_CHARACTERS_RE.sub("", str(v))[:max_str_len]


print("Preparing data for Excel export...")
df_x = meta_cluster_all
for field in df_x.schema.fields:
    dt = field.dataType
    if isinstance(dt, (T.ArrayType, T.MapType, T.StructType)):
        df_x = df_x.withColumn(field.name, F.to_json(F.col(field.name)))
    elif isinstance(dt, T.BinaryType):
        df_x = df_x.withColumn(field.name, F.base64(F.col(field.name)))

cols = df_x.columns

print("Writing to Excel workbook...")
wb = Workbook(write_only=True)
ws = wb.create_sheet(title="meta")
ws.append(cols)
for r in df_x.select(*cols).toLocalIterator():
    ws.append([_sanitize_cell(v) for v in list(r)])

wb.save(meta_cluster_xlsx)
print(f"Successfully exported to: {meta_cluster_xlsx}")
#"""

'\n# Prepare data for Export to Excel\n# ==========================================\nmeta_cluster_xlsx = "/home/jovyan/work/meta_cluster_w2v.xlsx"\nmax_str_len = 5000\n\ndef _sanitize_cell(v):\n    if v is None:\n        return None\n    if isinstance(v, str):\n        return ILLEGAL_CHARACTERS_RE.sub("", v)[:max_str_len]\n    if isinstance(v, (int, float, bool, date, datetime)):\n        return v\n    return ILLEGAL_CHARACTERS_RE.sub("", str(v))[:max_str_len]\n\n\nprint("Preparing data for Excel export...")\ndf_x = meta_cluster_all\nfor field in df_x.schema.fields:\n    dt = field.dataType\n    if isinstance(dt, (T.ArrayType, T.MapType, T.StructType)):\n        df_x = df_x.withColumn(field.name, F.to_json(F.col(field.name)))\n    elif isinstance(dt, T.BinaryType):\n        df_x = df_x.withColumn(field.name, F.base64(F.col(field.name)))\n\ncols = df_x.columns\n\nprint("Writing to Excel workbook...")\nwb = Workbook(write_only=True)\nws = wb.create_sheet(title="meta")\nws.append(cols)\nf

In [ ]:
# Review Filtering & Cluster Merging
# ==========================================
import os
from pyspark.sql import functions as F

# 1. Setup paths
review_in = review_out  # Cleaned reviews from your previous step
meta_cluster_in = "ecommerce/data_lake/cluster/meta"
review_cluster_out = "ecommerce/data_lake/cluster/review"

local_data_lake = "/home/jovyan/work/data_lake"
os.makedirs(local_data_lake, exist_ok=True)

meta_cluster_local_path = f"{local_data_lake}/meta_cluster.parquet"
review_cluster_local_path = f"{local_data_lake}/review_cluster.parquet"

print("Reading clustered metadata and cleaned reviews...")
df_meta_clustered = spark.read.parquet(meta_cluster_in)
df_review_raw = spark.read.parquet(review_in)

# 2. Extract ASIN-Cluster mapping (ensure distinctness to avoid data duplication)
# We only need parent_asin and the cluster label
asin_cluster_map = df_meta_clustered.select("parent_asin", "cluster").distinct()

# 3. Perform Broadcast Join
# Since asin_cluster_map is much smaller than raw reviews, 
# broadcasting it prevents massive Shuffle and OOM.
print("Merging cluster labels into review data via Broadcast Join...")
df_review_clustered = df_review_raw.join(
    F.broadcast(asin_cluster_map),
    on="parent_asin",
    how="inner"
)

# 4. Show Data Volume Statistics
meta_count = df_meta_clustered.count()
review_count = df_review_clustered.count()

print("\n" + "="*40)
print(f"DATA VOLUME SUMMARY:")
print(f"Total Clustered Metadata: {meta_count:,}")
print(f"Total Filtered & Labeled Reviews: {review_count:,}")
print("="*40 + "\n")

# 5. Persist to HDFS (Partitioned by cluster for M4's convenience)
print(f"Saving clustered reviews to HDFS: {review_cluster_out} ...")
df_review_clustered.write \
    .mode("overwrite") \
    .partitionBy("cluster") \
    .parquet(review_cluster_out)

Reading clustered metadata and cleaned reviews...
Merging cluster labels into review data via Broadcast Join...

DATA VOLUME SUMMARY:
Total Clustered Metadata: 131,142
Total Filtered & Labeled Reviews: 6,272,751

Saving clustered reviews to HDFS: ecommerce/data_lake/cluster/review ...


In [ ]:
"""
# Optional: Output the clusters
# ==========================================
# Member 2: Export to Local Data Lake (for local inspection/for M3 & M4 download)
print("Exporting results to local file system...")

# Saving metadata to local
df_meta_clustered.write.mode("overwrite").parquet(f"file://{meta_cluster_local_path}")

# Saving review data to local
# Note: For massive data, coalesce(1) is slow; 
# but if you need a manageable local folder, it helps.
df_review_clustered.write.mode("overwrite").parquet(f"file://{review_cluster_local_path}")

print("\nAll tasks completed!")
print(f"Local Meta Path: {meta_cluster_local_path}")
print(f"Local Review Path: {review_cluster_local_path}")
# """

'\n# Optional: Output the clusters\n# ==========================================\n# Member 2: Export to Local Data Lake (for local inspection/for M3 & M4 download)\nprint("Exporting results to local file system...")\n\n# Saving metadata to local\ndf_meta_clustered.write.mode("overwrite").parquet(f"file://{meta_cluster_local_path}")\n\n# Saving review data to local\n# Note: For massive data, coalesce(1) is slow; \n# but if you need a manageable local folder, it helps.\ndf_review_clustered.write.mode("overwrite").parquet(f"file://{review_cluster_local_path}")\n\nprint("\nAll tasks completed!")\nprint(f"Local Meta Path: {meta_cluster_local_path}")\nprint(f"Local Review Path: {review_cluster_local_path}")\n# '

# Phase 4 Data Analytics

## Member 3
### Big Data Analyst

In [ ]:
# Read the cluster metadata from local or HDFS Data Lake
# Data Initialization & Loading (Metadata)
# ==========================================
import os
from pyspark.sql import functions as F

print("--- Data Initialization ---")

import os

local_data_lake = "/home/jovyan/work/data_lake"
meta_cluster_local_path = f"{local_data_lake}/meta_cluster.parquet"
meta_cluster_local_uri = f"file://{meta_cluster_local_path}"

hdfs_namenode = os.environ.get("HDFS_NAMENODE", "hdfs://namenode:9000")
meta_lake_path = "ecommerce/data_lake/cluster/meta"
meta_hdfs_candidates = [
    f"{hdfs_namenode}/user/root/{meta_lake_path}",
    f"{hdfs_namenode}/{meta_lake_path}",
]

def hdfs_exists(hdfs_path: str) -> bool:
    jvm = spark._jvm
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(
        jvm.java.net.URI.create(hdfs_path),
        spark._jsc.hadoopConfiguration()
    )
    return fs.exists(jvm.org.apache.hadoop.fs.Path(hdfs_path))

def read_parquet_local_fallback_hdfs(local_path: str, local_uri: str, hdfs_candidates):
    if os.path.exists(local_path):
        try:
            return spark.read.parquet(local_uri), local_uri
        except Exception as e:
            print(f"Local parquet read failed, fallback to HDFS. local={local_path}, err={e}")

    for hdfs_path in hdfs_candidates:
        try:
            if hdfs_exists(hdfs_path):
                return spark.read.parquet(hdfs_path), hdfs_path
        except Exception:
            pass

    raise FileNotFoundError(f"Neither local nor HDFS path exists. local={local_path} hdfs_candidates={hdfs_candidates}")

df_meta, loaded_from = read_parquet_local_fallback_hdfs(meta_cluster_local_path, meta_cluster_local_uri, meta_hdfs_candidates)
print(f"Loaded clustered metadata from: {loaded_from}")

# 3. (Optional) Persist to local HDFS/Data Lake for subsequent faster reads
# Uncomment the following lines if you want to build the local data lake structure
"""
print(f"Persisting data to local Data Lake: {meta_lake_path}")
df_meta.write \
    .mode("overwrite") \
    .partitionBy("cluster") \
    .parquet(meta_lake_path)
"""

# 4. Verification
total_items = df_meta.count()
print(f"Successfully loaded {total_items:,} items.")
print("Sample Data Structure:")
df_meta.show(5, truncate=False)

print("Data is ready for Premium/Price Analysis!")


--- Data Initialization ---
Loaded clustered metadata from: file:///home/jovyan/work/data_lake/meta_cluster.parquet
Successfully loaded 131,142 items.
Sample Data Structure:
+------+--------------+---------------+----------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# Advanced Brand Premium Analysis
# ==========================================
# used spark.sql

print("\n--- Starting Advanced Brand Premium Analysis ---")
df_meta.createOrReplaceTempView("meta_table")

# 1. Basic Brand Pricing (Direct GroupBy Store)
# To answer: What is the overall average price of each brand/store?
print("1. Macro Overview: Overall Brand Pricing")
basic_brand_df = spark.sql("""
    SELECT 
        store,
        COUNT(*) as total_products,
        ROUND(AVG(price), 2) as overall_avg_price
    FROM meta_table
    WHERE price IS NOT NULL AND store IS NOT NULL
    GROUP BY store
    HAVING COUNT(*) > 50  -- Filter out niche sellers
    ORDER BY overall_avg_price DESC
""")
basic_brand_df.show(10, truncate=False)


# 2. True Brand Premium (Store inside specific Clusters)
# To answer: For products in the SAME configuration tier (cluster), which brand charges more?
print("2. Deep Dive: True Brand Premium within the Same Configuration Tier")
true_premium_df = spark.sql("""
    SELECT 
        cluster,
        store,
        COUNT(*) as product_count,
        ROUND(AVG(price), 2) as tier_avg_price,
        ROUND(MAX(price), 2) as tier_max_price
    FROM meta_table
    WHERE price IS NOT NULL AND store IS NOT NULL
    GROUP BY cluster, store
    HAVING COUNT(*) > 20  -- Ensure statistical significance within the cluster
    ORDER BY cluster ASC, tier_avg_price DESC
""")

true_premium_df.show(20, truncate=False)

local_data_lake = "/home/jovyan/work/data_lake"
output_path = f"{local_data_lake}/true_brand_premium.csv"

# Optional: Export this data for Visualization Expert
true_premium_pandas = true_premium_df.toPandas()
true_premium_pandas.to_csv(output_path, index=False)
print("Brand Premium SQL Analysis complete. Ready for visualization.")


--- Starting Advanced Brand Premium Analysis ---
1. Macro Overview: Overall Brand Pricing
+---------+--------------+-----------------+
|store    |total_products|overall_avg_price|
+---------+--------------+-----------------+
|Leica    |175           |1988.51          |
|Alienware|56            |1754.4           |
|Apple    |59            |954.37           |
|MSI      |376           |751.74           |
|msi      |368           |717.91           |
|Sigma    |389           |691.49           |
|BenQ     |113           |667.12           |
|Canon    |1542          |603.78           |
|ASUS     |2176          |570.49           |
|Lenovo   |2899          |561.11           |
+---------+--------------+-----------------+
only showing top 10 rows

2. Deep Dive: True Brand Premium within the Same Configuration Tier
+-------+---------+-------------+--------------+--------------+
|cluster|store    |product_count|tier_avg_price|tier_max_price|
+-------+---------+-------------+--------------+---------

In [ ]:
""" Legacy Code
from pyspark.sql import functions as F

pricing_trends_out = "/ecommerce/warehouse/pricing_trends"

meta = spark.read.parquet(meta_cluster_out)
reviews = spark.read.parquet(review_out)

def with_year_month(df):
    if "year_month" in df.columns:
        return df
    if "unixReviewTime" in df.columns:
        ts = F.col("unixReviewTime").cast("long")
    elif "timestamp" in df.columns:
        ts = F.col("timestamp").cast("long")
    elif "time" in df.columns:
        ts = F.col("time").cast("long")
    elif "reviewTime" in df.columns:
        dt = F.to_date(F.col("reviewTime"))
        return df.withColumn("review_date", dt).withColumn("year_month", F.date_format(dt, "yyyy-MM"))
    else:
        return df.withColumn("year_month", F.lit(None).cast("string"))

    ts_sec = F.when(ts > F.lit(20000000000), (ts / 1000).cast("long")).otherwise(ts)
    return df.withColumn("review_date", F.from_unixtime(ts_sec).cast("date")).withColumn("year_month", F.date_format(F.col("review_date"), "yyyy-MM"))

reviews = with_year_month(reviews)

if "overall" not in reviews.columns and "rating" in reviews.columns:
    reviews = reviews.withColumn("overall", F.col("rating").cast("double"))

join_key_candidates = ["asin", "parent_asin", "ASIN", "product_id"]
join_key = next((k for k in join_key_candidates if k in reviews.columns and k in meta.columns), None)

base = reviews
if join_key:
    meta_cols = [join_key] + (["price"] if "price" in meta.columns else [])
    reviews_cols = [join_key] + (["overall"] if "overall" in reviews.columns else []) + (["year_month"] if "year_month" in reviews.columns else [])
    base = reviews.select(reviews_cols).join(meta.select(meta_cols), on=join_key, how="left")

agg_exprs = [F.count("*").alias("review_count")]
agg_exprs.append(F.countDistinct(join_key).alias("distinct_products") if join_key else F.lit(None).cast("long").alias("distinct_products"))
agg_exprs.append(F.avg("overall").alias("avg_rating") if "overall" in base.columns else F.lit(None).cast("double").alias("avg_rating"))
agg_exprs.append(F.avg("price").alias("avg_price") if "price" in base.columns else F.lit(None).cast("double").alias("avg_price"))

trends = base.groupBy("year_month").agg(*agg_exprs).orderBy("year_month")

trends.write.mode("overwrite").parquet(pricing_trends_out)
print(pricing_trends_out)
trends.show(24, truncate=False)
#"""

' Legacy Code\nfrom pyspark.sql import functions as F\n\npricing_trends_out = "/ecommerce/warehouse/pricing_trends"\n\nmeta = spark.read.parquet(meta_cluster_out)\nreviews = spark.read.parquet(review_out)\n\ndef with_year_month(df):\n    if "year_month" in df.columns:\n        return df\n    if "unixReviewTime" in df.columns:\n        ts = F.col("unixReviewTime").cast("long")\n    elif "timestamp" in df.columns:\n        ts = F.col("timestamp").cast("long")\n    elif "time" in df.columns:\n        ts = F.col("time").cast("long")\n    elif "reviewTime" in df.columns:\n        dt = F.to_date(F.col("reviewTime"))\n        return df.withColumn("review_date", dt).withColumn("year_month", F.date_format(dt, "yyyy-MM"))\n    else:\n        return df.withColumn("year_month", F.lit(None).cast("string"))\n\n    ts_sec = F.when(ts > F.lit(20000000000), (ts / 1000).cast("long")).otherwise(ts)\n    return df.withColumn("review_date", F.from_unixtime(ts_sec).cast("date")).withColumn("year_month", F.d

# Member 4
## Machine Learning Engineer

In [ ]:
# Data Initialization & Feature Fusion (Review + Meta)
# ==========================================
import os
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

print("--- Data Initialization & Feature Fusion ---")

local_data_lake = "/home/jovyan/work/data_lake"
meta_cluster_local_path = f"{local_data_lake}/meta_cluster.parquet"
meta_cluster_local_uri = f"file://{meta_cluster_local_path}"
review_cluster_local_path = f"{local_data_lake}/review_cluster.parquet"
review_cluster_local_uri = f"file://{review_cluster_local_path}"

hdfs_namenode = os.environ.get("HDFS_NAMENODE", "hdfs://namenode:9000")
meta_lake_path = "ecommerce/data_lake/cluster/meta"
review_lake_path = "ecommerce/data_lake/cluster/review"
meta_hdfs_candidates = [
    f"{hdfs_namenode}/user/root/{meta_lake_path}",
    f"{hdfs_namenode}/{meta_lake_path}",
]

review_hdfs_candidates = [
    f"{hdfs_namenode}/user/root/{review_lake_path}",
    f"{hdfs_namenode}/{review_lake_path}",
]

def hdfs_exists(hdfs_path: str) -> bool:
    jvm = spark._jvm
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(
        jvm.java.net.URI.create(hdfs_path),
        spark._jsc.hadoopConfiguration()
    )
    return fs.exists(jvm.org.apache.hadoop.fs.Path(hdfs_path))

def read_parquet_local_fallback_hdfs(local_path: str, local_uri: str, hdfs_candidates):
    if os.path.exists(local_path):
        try:
            return spark.read.parquet(local_uri), local_uri
        except Exception as e:
            print(f"Local parquet read failed, fallback to HDFS. local={local_path}, err={e}")

    for hdfs_path in hdfs_candidates:
        try:
            if hdfs_exists(hdfs_path):
                return spark.read.parquet(hdfs_path), hdfs_path
        except Exception:
            pass

    raise FileNotFoundError(f"Neither local nor HDFS path exists. local={local_path} hdfs_candidates={hdfs_candidates}")

df_meta, loaded_from = read_parquet_local_fallback_hdfs(meta_cluster_local_path, meta_cluster_local_uri, meta_hdfs_candidates)
print(f"Loaded clustered metadata from: {loaded_from}")

df_review, loaded_from = read_parquet_local_fallback_hdfs(review_cluster_local_path, review_cluster_local_uri, review_hdfs_candidates)
print(f"Loaded clustered review data from: {loaded_from}")

# 3. Extract necessary features from Metadata 
# Added price, and filtered out null prices.
print("Extracting features (store, price) from Metadata...") # Cluster is already in review
meta_features = df_meta.select("parent_asin", "store", "price") \
                       .filter(F.col("price").isNotNull()) \
                       .distinct()

# 4. Merge Meta features into Reviews (Broadcast Join for performance)
print("Fusing metadata features into review dataset...")
df_review_enriched = df_review.join(
    F.broadcast(meta_features),
    on="parent_asin",
    how="inner"
)

# 5. Create Binary Label for Machine Learning (Sentiment Analysis)
# Assumption: Rating 4-5 is Positive (1.0), 1-3 is Negative (0.0)
df_review_enriched = df_review_enriched.withColumn(
    "label", 
    F.when(F.col("rating") >= 4, 1.0).otherwise(0.0)
)

# Verification
review_count = df_review_enriched.count()
print(f"Successfully loaded and enriched {review_count:,} reviews (with valid pricing).")
df_review_enriched.show(5)

print("Data is ready for NLP Sentiment Analysis!")

--- Data Initialization & Feature Fusion ---
Loaded clustered metadata from: file:///home/jovyan/work/data_lake/meta_cluster.parquet
Loaded clustered review data from: file:///home/jovyan/work/data_lake/review_cluster.parquet
Extracting features (store, price) from Metadata...
Fusing metadata features into review dataset...
Successfully loaded and enriched 3,516,083 reviews (with valid pricing).
+-----------+----------+------+------------+--------------------+-------------+------------+-----------------+--------------------+-------+--------------------+--------------------+--------------+-----------+----------+-------+--------+------+-----+
|parent_asin|      asin|rating|       title|                text|    timestamp|helpful_vote|verified_purchase|             user_id|overall|          reviewText|          reviewerID|unixReviewTime|review_date|year_month|cluster|   store| price|label|
+-----------+----------+------+------------+--------------------+-------------+------------+---------

In [ ]:
# Advanced Sentiment Analysis
# ==========================================
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression

print("\n--- Starting Spark MLlib Sentiment Analysis ---")

# 1. Ensure text column is clean (dropping nulls in text)
df_model_data = df_review_enriched.filter(F.col("text").isNotNull())

# 2. Build the NLP Pipeline
print("Building NLP Pipeline (Tokenizer -> StopWords -> TF-IDF)...")
tokenizer = Tokenizer(inputCol="text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=10000)
idf = IDF(inputCol="rawFeatures", outputCol="features")

# 3. Define the Classifier (Logistic Regression as per ACM spec)
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# 4. Assemble and Train the Pipeline
pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, lr])

# Split data (80% training, 20% testing)
print("Splitting data into Training (80%) and Testing (20%)...")
train_data, test_data = df_model_data.randomSplit([0.8, 0.2], seed=42)

print("Training Logistic Regression Model... (This might take a while)")
model = pipeline.fit(train_data)

# 5. Generate Predictions on Test Data
print("Generating sentiment predictions...")
predictions = model.transform(test_data)

# Show sample predictions (probability and predicted label)
predictions.select("text", "price", "store", "label", "prediction").show(5, truncate=True)

# ==========================================
# Phase 4 Business Insights: Sentiment vs. Price/Store
# ==========================================
print("\n--- Deep Analytics: Sentiment Inflection Points ---")

# Register predictions as a temporary view for SQL analytics
predictions.createOrReplaceTempView("sentiment_predictions")

# 1. Store Performance: Which store has the highest sentiment score?
print("1. Sentiment Analysis by Store (Top 10):")
store_sentiment_df = spark.sql("""
    SELECT 
        store,
        COUNT(*) as review_count,
        ROUND(AVG(price), 2) as avg_price,
        ROUND(AVG(prediction), 4) as positive_sentiment_ratio
    FROM sentiment_predictions
    WHERE store IS NOT NULL
    GROUP BY store
    HAVING review_count > 50
    ORDER BY positive_sentiment_ratio DESC
""")
store_sentiment_df.show(10, truncate=False)

# 2. Pricing Inflection Points: How does price affect sentiment?
# We create $100 price buckets to see where the sentiment drops
print("2. Pricing Inflection Points (Sentiment by $100 Price Buckets):")
price_sentiment_df = spark.sql("""
    SELECT 
        FLOOR(price / 100) * 100 as price_bucket,
        COUNT(*) as volume,
        ROUND(AVG(prediction), 4) as avg_sentiment
    FROM sentiment_predictions
    GROUP BY price_bucket
    HAVING volume > 20
    ORDER BY price_bucket ASC
""")
price_sentiment_df.show(20, truncate=False)


--- Starting Spark MLlib Sentiment Analysis ---
Building NLP Pipeline (Tokenizer -> StopWords -> TF-IDF)...
Splitting data into Training (80%) and Testing (20%)...
Training Logistic Regression Model... (This might take a while)


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


Py4JError: An error occurred while calling o2435.fit

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [ ]:
# Business Insights: Sentiment vs. Price/Store/Cluster
# ==========================================
print("\n--- Deep Analytics: Sentiment Inflection Points by Configuration Tier ---")

# Register predictions as a temporary view for SQL analytics
# Note: 'cluster' is implicitly available here from the enriched dataframe
predictions.createOrReplaceTempView("sentiment_predictions")

# 1. Store Performance within Clusters: Which store performs best in each tier?
print("1. Sentiment Analysis by Store and Cluster:")
store_cluster_sentiment_df = spark.sql("""
    SELECT 
        cluster,
        store,
        COUNT(*) as review_count,
        ROUND(AVG(price), 2) as avg_price,
        ROUND(AVG(prediction), 4) as positive_sentiment_ratio
    FROM sentiment_predictions
    WHERE store IS NOT NULL
    GROUP BY cluster, store
    HAVING review_count > 20
    ORDER BY cluster ASC, positive_sentiment_ratio DESC
""")
store_cluster_sentiment_df.show(20, truncate=False)

# 2. Pricing Inflection Points per Cluster: 
# Finding where sentiment drops for EACH configuration tier
print("2. Pricing Inflection Points by Cluster (Sentiment by $100 Price Buckets):")
price_cluster_sentiment_df = spark.sql("""
    SELECT 
        cluster,
        FLOOR(price / 100) * 100 as price_bucket,
        COUNT(*) as volume,
        ROUND(AVG(prediction), 4) as avg_sentiment
    FROM sentiment_predictions
    GROUP BY cluster, price_bucket
    HAVING volume > 15
    ORDER BY cluster ASC, price_bucket ASC
""")
price_cluster_sentiment_df.show(30, truncate=False)

local_data_lake = "/home/jovyan/work/data_lake"
# 3. Export Data for Visualization (Optional but recommended for the final report)
print("\nExporting aggregated data for visualization...")
store_cluster_sentiment_df.toPandas().to_csv(f"{local_data_lake}/store_sentiment_by_cluster.csv", index=False)
price_cluster_sentiment_df.toPandas().to_csv(f"{local_data_lake}/price_inflection_by_cluster.csv", index=False)
print("Phase 4 Pipeline execution complete!")

In [ ]:
""" Legacy Code

from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import HashingTF, IDF, StopWordsRemover, Tokenizer

sentiment_monthly_out = "/ecommerce/warehouse/sentiment_monthly"
model_out = "/ecommerce/models/sentiment_lr"

reviews_full = spark.read.parquet(review_out)

if "reviewText" not in reviews_full.columns and "text" in reviews_full.columns:
    reviews_full = reviews_full.withColumn("reviewText", F.col("text").cast("string"))
if "overall" not in reviews_full.columns and "rating" in reviews_full.columns:
    reviews_full = reviews_full.withColumn("overall", F.col("rating").cast("double"))

def to_unix_seconds(col):
    ts = col.cast("long")
    return F.when(ts > F.lit(20000000000), (ts / 1000).cast("long")).otherwise(ts)

def with_year_month(df):
    if "year_month" in df.columns:
        return df
    if "unixReviewTime" in df.columns:
        ts = to_unix_seconds(F.col("unixReviewTime"))
    elif "timestamp" in df.columns:
        ts = to_unix_seconds(F.col("timestamp"))
    elif "time" in df.columns:
        ts = to_unix_seconds(F.col("time"))
    elif "reviewTime" in df.columns:
        dt = F.to_date(F.col("reviewTime"))
        return df.withColumn("review_date", dt).withColumn("year_month", F.date_format(dt, "yyyy-MM"))
    else:
        return df.withColumn("year_month", F.lit(None).cast("string"))

    return df.withColumn("review_date", F.from_unixtime(ts).cast("date")).withColumn("year_month", F.date_format(F.col("review_date"), "yyyy-MM"))

reviews_full = with_year_month(reviews_full)

train_df = reviews_full.filter(
    F.col("reviewText").isNotNull() & (F.length(F.col("reviewText")) > 0) & F.col("overall").isNotNull()
)
train_df = train_df.withColumn(
    "label",
    F.when(F.col("overall") >= 4, F.lit(1.0)).when(F.col("overall") <= 2, F.lit(0.0))
)
train_df = train_df.filter(F.col("label").isNotNull()).select("reviewText", "label")

tokenizer = Tokenizer(inputCol="reviewText", outputCol="tokens")
remover = StopWordsRemover(inputCol="tokens", outputCol="filtered")
tf = HashingTF(inputCol="filtered", outputCol="tf", numFeatures=262144)
idf = IDF(inputCol="tf", outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20, regParam=0.1)

pipeline = Pipeline(stages=[tokenizer, remover, tf, idf, lr])
train, test = train_df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)
pred = model.transform(test)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = evaluator.evaluate(pred)
print(f"AUC: {auc}")

scored = model.transform(reviews_full.filter(F.col("reviewText").isNotNull() & (F.length(F.col("reviewText")) > 0)))

try:
    from pyspark.ml.functions import vector_to_array
    scored = scored.withColumn("sentiment_score", vector_to_array(F.col("probability"))[1])
except Exception:
    from pyspark.sql.types import DoubleType
    scored = scored.withColumn("sentiment_score", F.udf(lambda v: float(v[1]) if v is not None else None, DoubleType())(F.col("probability")))

sent_month = scored.groupBy("year_month").agg(
    F.count("*").alias("scored_reviews"),
    F.avg("sentiment_score").alias("avg_sentiment")
).orderBy("year_month")

sent_month.write.mode("overwrite").parquet(sentiment_monthly_out)
model.write().overwrite().save(model_out)

print(sentiment_monthly_out)
print(model_out)
sent_month.show(24, truncate=False)#"""

## Member 5
### Visuailization Expert

In [ ]:
# Final Report Dashboards & Visualizations
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

print("--- Initializing Stratified Dashboards for ACM Report ---")
sns.set_theme(style="whitegrid", palette="muted")

local_data_lake = "/home/jovyan/work/data_lake"

# ==========================================
# SMART FILTERING MODULE (Stratified Selection)
# ==========================================
try:
    df_base = pd.read_csv(f"{local_data_lake}/true_brand_premium.csv")
    
    # Calculate price variance for each cluster to identify "High Interest" cases
    cluster_variance = df_base.groupby('cluster')['tier_avg_price'].std().fillna(0)
    
    # Define the required category order
    target_categories = ['Ce', 'Ca', 'Co', 'El', 'Ha']
    TARGET_CLUSTERS = []
    
    # Pick top 2 highest variance clusters for each category to ensure 10 plots
    for cat in target_categories:
        # Match clusters starting with the prefix (e.g., 'Ce_')
        cat_clusters = [c for c in cluster_variance.index if str(c).startswith(cat)]
        # Sort by variance descending and take top 2
        top_2 = sorted(cat_clusters, key=lambda c: cluster_variance[c], reverse=True)[:2]
        TARGET_CLUSTERS.extend(top_2)
        
    print(f"[*] Selected 10 Clusters (Ordered): {TARGET_CLUSTERS}")

except FileNotFoundError:
    print(f"Error: Base CSV not found in {local_data_lake}.")
    TARGET_CLUSTERS = []

# ==========================================
# 1. Chart 1: True Brand Premium
# ==========================================
if TARGET_CLUSTERS:
    print("\n1. Generating True Brand Premium Chart...")
    df_premium = pd.read_csv(f"{local_data_lake}/true_brand_premium.csv")
    
    # Apply stratified filter
    df_premium = df_premium[df_premium['cluster'].isin(TARGET_CLUSTERS)]
    
    # Select top 5 stores per cluster for clarity
    top_stores_df = df_premium.groupby('cluster', group_keys=False).apply(
        lambda x: x.nlargest(5, 'product_count')
    ).reset_index(drop=True)

    g = sns.catplot(
        data=top_stores_df,
        kind="bar",
        x="store",
        y="tier_avg_price",
        col="cluster",
        col_order=TARGET_CLUSTERS,    # <--- [FIX 1] Enforces the Ce->Ca->Co->El->Ha order
        col_wrap=3,                   # <--- [FIX 2] Forces 3 columns per row
        sharex=False,         
        sharey=True,          
        palette="viridis",
        height=3.2,                   # <--- [FIX 3] Reduced height so 2 rows fit on one screen
        aspect=1.1,          
        legend=False          
    )
    
    g.fig.suptitle("True Brand Premium: High-Variance Clusters across Categories", 
                   fontsize=18, fontweight='bold', y=1.05)
    g.set_axis_labels("Store", "Avg Price (USD)")
    g.set_titles("Cluster {col_name}")
    
    for ax in g.axes.flat:
        for label in ax.get_xticklabels():
            label.set_rotation(45)
            label.set_ha('right')
            
    plt.tight_layout()
    plt.show()

# ==========================================
# 2. Chart 2: Store Sentiment Performance (Filtered)
# ==========================================
if TARGET_CLUSTERS:
    print("\n2. Generating Store Sentiment Performance Chart...")
    df_store_sent = pd.read_csv(f"{local_data_lake}/store_sentiment_by_cluster.csv")
    df_store_sent = df_store_sent[df_store_sent['cluster'].isin(TARGET_CLUSTERS)]
    
    plt.figure(figsize=(12, 6))
    ax2 = sns.scatterplot(
        data=df_store_sent, 
        x='avg_price', 
        y='positive_sentiment_ratio', 
        hue='cluster', 
        hue_order=TARGET_CLUSTERS,   # Keep color consistent with the order
        size='review_count', 
        sizes=(100, 800), 
        alpha=0.75,
        palette='deep'
    )
    plt.title("Store Sentiment Analysis (Selected Stratified Clusters)", fontsize=16, fontweight='bold', pad=15)
    plt.xlabel("Average Store Price (USD)", fontsize=12, fontweight='bold')
    plt.ylabel("Positive Sentiment Ratio", fontsize=12, fontweight='bold')
    
    plt.axhline(0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='50% Threshold')
    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=5, frameon=False)
    plt.tight_layout()
    plt.show()

# ==========================================
# 3. Chart 3: Pricing Inflection Points (Filtered)
# ==========================================
if TARGET_CLUSTERS:
    print("\n3. Generating Pricing Inflection Points Chart...")
    df_inflection = pd.read_csv(f"{local_data_lake}/price_inflection_by_cluster.csv")
    df_inflection = df_inflection[df_inflection['cluster'].isin(TARGET_CLUSTERS)]
    
    plt.figure(figsize=(14, 7))
    ax3 = sns.lineplot(
        data=df_inflection, 
        x='price_bucket', 
        y='avg_sentiment', 
        hue='cluster', 
        hue_order=TARGET_CLUSTERS,   # Keep lines in the same category order
        marker='o', 
        markersize=8, 
        linewidth=2.5,
        palette='bright'
    )
    plt.title("Pricing Inflection Points: Sentiment Decay per Category", fontsize=16, fontweight='bold', pad=15)
    plt.xlabel("Price Bucket (USD)", fontsize=12, fontweight='bold')
    plt.ylabel("Average Sentiment Ratio", fontsize=12, fontweight='bold')
    
    plt.axhline(0.8, color='green', linestyle=':', linewidth=1.2, alpha=0.6, label='Satisfied (80%)')
    plt.axhline(0.5, color='red', linestyle='--', linewidth=1.2, alpha=0.6, label='Inflection (50%)')
    
    plt.grid(True, which='both', linestyle='--', alpha=0.5)
    plt.legend(title='Selected Clusters', loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=5, frameon=False)
    plt.tight_layout()
    plt.show()

print("Stratified charts generated successfully!")

In [ ]:
# Try Try...
# Final Report Dashboards & Visualizations (Smart Filtered Version)
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

print("--- Initializing Smart Dashboards for ACM Report ---")
sns.set_theme(style="whitegrid", palette="muted")

local_data_lake = "/home/jovyan/work/data_lake"

# ==========================================
# SMART FILTERING MODULE
# ==========================================
try:
    # We use true_brand_premium as the baseline to calculate cluster metrics
    df_base = pd.read_csv(f"{local_data_lake}/true_brand_premium.csv")
    
    # 1. Identify (High Variance Clusters)
    # Standard deviation of price within each cluster
    cluster_variance = df_base.groupby('cluster')['tier_avg_price'].std().fillna(0)
    top_variance_clusters = cluster_variance.nlargest(9).index.tolist()
    
    # 2. Identify (High Average Price Clusters)
    # Mean price of each cluster
    cluster_mean = df_base.groupby('cluster')['tier_avg_price'].mean()
    top_value_clusters = cluster_mean.nlargest(9).index.tolist()
    
    print(f"[*] Analysis - High Variance Clusters: {top_variance_clusters}")
    print(f"[*] Analysis - High Value Clusters: {top_value_clusters}")
    
    # ========================================================
    # top_variance_clusters
    # top_value_clusters
    # ========================================================
    TARGET_CLUSTERS = top_variance_clusters  
    
    print(f"\n[!] Filtering all charts to strictly display Clusters: {TARGET_CLUSTERS}")

except FileNotFoundError:
    print(f"Error: Base CSV not found in {local_data_lake}.")
    TARGET_CLUSTERS = []

# ==========================================
# 1. Chart 1: True Brand Premium (Filtered)
# ==========================================
if TARGET_CLUSTERS:
    print("\n1. Generating True Brand Premium Chart (Filtered)...")
    df_premium = pd.read_csv(f"{local_data_lake}/true_brand_premium.csv")
    
    # APPLY FILTER
    df_premium = df_premium[df_premium['cluster'].isin(TARGET_CLUSTERS)]
    
    top_stores_df = df_premium.groupby('cluster', group_keys=False).apply(
        lambda x: x.nlargest(5, 'product_count')
    ).reset_index(drop=True)

    g = sns.catplot(
        data=top_stores_df,
        kind="bar",
        x="store",
        y="tier_avg_price",
        col="cluster",
        col_wrap=3,           # <--- [CHANGE 1] Change from len(TARGET_CLUSTERS) to 3. This forces the 3x3 grid.
        sharex=False,         
        sharey=True,          
        palette="viridis",
        height=3.5,           # <--- [CHANGE 2] Change from 4.5 to 3.5. This prevents the 9 plots from stretching off the screen.
        aspect=1.2,
        legend=False          
    )
    
    g.fig.suptitle("True Brand Premium (Selected High-Interest Clusters)", 
                   fontsize=16, fontweight='bold', y=1.05)
    g.set_axis_labels("Store", "Average Tier Price (USD)")
    g.set_titles("Cluster {col_name}")
    
    for ax in g.axes.flat:
        for label in ax.get_xticklabels():
            label.set_rotation(45)
            label.set_ha('right')
            
    plt.tight_layout()
    plt.show()

# ==========================================
# 2. Chart 2: Store Sentiment Performance (Filtered)
# ==========================================
if TARGET_CLUSTERS:
    print("\n2. Generating Store Sentiment Performance Chart (Filtered)...")
    df_store_sent = pd.read_csv(f"{local_data_lake}/store_sentiment_by_cluster.csv")
    
    # APPLY FILTER
    df_store_sent = df_store_sent[df_store_sent['cluster'].isin(TARGET_CLUSTERS)]
    
    plt.figure(figsize=(10, 6))
    ax2 = sns.scatterplot(
        data=df_store_sent, 
        x='avg_price', 
        y='positive_sentiment_ratio', 
        hue='cluster', 
        size='review_count', 
        sizes=(150, 900), 
        alpha=0.75,
        palette='deep',
        edgecolor='w',
        linewidth=1
    )
    plt.title("Store Sentiment Analysis (Selected Clusters)", fontsize=16, fontweight='bold', pad=15)
    plt.xlabel("Average Store Price (USD)", fontsize=12, fontweight='bold')
    plt.ylabel("Positive Sentiment Ratio (0.0 to 1.0)", fontsize=12, fontweight='bold')
    
    plt.axhline(0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='50% Sentiment Threshold')
    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4, frameon=False)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 3. Chart 3: Pricing Inflection Points (Filtered)
# ==========================================
if TARGET_CLUSTERS:
    print("\n3. Generating Pricing Inflection Points Chart (Filtered)...")
    df_inflection = pd.read_csv(f"{local_data_lake}/price_inflection_by_cluster.csv")
    
    # APPLY FILTER
    df_inflection = df_inflection[df_inflection['cluster'].isin(TARGET_CLUSTERS)]
    
    plt.figure(figsize=(12, 6))
    ax3 = sns.lineplot(
        data=df_inflection, 
        x='price_bucket', 
        y='avg_sentiment', 
        hue='cluster', 
        marker='o', 
        markersize=10, 
        linewidth=3,
        palette='bright'
    )
    plt.title("Pricing Inflection Points (Selected Clusters)", fontsize=16, fontweight='bold', pad=15)
    plt.xlabel("Price Bucket (USD)", fontsize=12, fontweight='bold')
    plt.ylabel("Average Sentiment Ratio", fontsize=12, fontweight='bold')
    
    plt.axhline(0.8, color='green', linestyle=':', linewidth=1.5, alpha=0.6, label='High Satisfaction (80%)')
    plt.axhline(0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.6, label='Critical Inflection (50%)')
    
    plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
    plt.legend(title='Selected Clusters', loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4, frameon=False)
    
    plt.tight_layout()
    plt.show()

print("Filtered charts generated successfully!")

In [ ]:
""" Legacy Code
# ==========================================
import matplotlib.pyplot as plt

pricing_trends_out = "/ecommerce/warehouse/pricing_trends"
sentiment_monthly_out = "/ecommerce/warehouse/sentiment_monthly"

pricing_df = spark.read.parquet(pricing_trends_out)
sent_df = spark.read.parquet(sentiment_monthly_out)

joined = pricing_df.join(sent_df, on="year_month", how="left").orderBy("year_month")
rows = joined.select("year_month", "avg_price", "review_count", "avg_sentiment").collect()

xm = [r["year_month"] for r in rows]
avg_price = [float(r["avg_price"]) if r["avg_price"] is not None else None for r in rows]
review_cnt = [int(r["review_count"]) for r in rows]
avg_sent = [float(r["avg_sentiment"]) if r["avg_sentiment"] is not None else None for r in rows]

positions = list(range(len(xm)))
n = len(positions)
stride = max(1, n // 12)
tick_pos = positions[::stride]
tick_labels = [xm[i] for i in tick_pos]
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(positions, avg_price, label="Avg Price")
ax1.set_xticks(tick_pos)
ax1.set_xticklabels(tick_labels, rotation=45, ha="right")
ax1.set_xlabel("Year-Month")
ax1.set_ylabel("Avg Price")

ax2 = ax1.twinx()
if any(v is not None for v in avg_sent):
    ax2.plot(positions, avg_sent, color="tab:orange", label="Avg Sentiment")
    ax2.set_ylabel("Avg Sentiment")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(positions, review_cnt)
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, rotation=45, ha="right")
ax.set_xlabel("Year-Month")
ax.set_ylabel("Review Count")
plt.tight_layout()
plt.show() 
#"""

In [ ]:
""" Legacy Code

from datetime import date, datetime
from openpyxl import Workbook
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
from pyspark.sql import functions as F
from pyspark.sql import types as T

meta_src = "/ecommerce/data_lake/meta/*"
out_path = "/home/jovyan/work/metadata_full_all_columns.xlsx"

max_str_len = 32767

def _sanitize_cell(v):
    if v is None:
        return None
    if isinstance(v, str):
        return ILLEGAL_CHARACTERS_RE.sub("", v)[:max_str_len]
    if isinstance(v, (int, float, bool, date, datetime)):
        return v
    return ILLEGAL_CHARACTERS_RE.sub("", str(v))[:max_str_len]

df = spark.read.parquet(meta_src)

# Convert non-scalar columns to JSON strings so they can be written into Excel
for field in df.schema.fields:
    dt = field.dataType
    if isinstance(dt, (T.ArrayType, T.MapType, T.StructType)):
        df = df.withColumn(field.name, F.to_json(F.col(field.name)))
    elif isinstance(dt, T.BinaryType):
        df = df.withColumn(field.name, F.base64(F.col(field.name)))

cols = df.columns
df = df.select(*cols)
total = df.count()
print(f"Exporting metadata rows: {total:,}")
print(f"Columns ({len(cols)}): {cols}")

wb = Workbook(write_only=True)
ws = wb.create_sheet(title="meta")
ws.append(cols)
written = 0
for r in df.toLocalIterator():
    ws.append([_sanitize_cell(v) for v in list(r)])
    written += 1
    if written % 50000 == 0:
        print(f"Wrote rows: {written:,}")

wb.save(out_path)
print(out_path)
#"""

In [ ]:
import time
time_end = time.time()

print(f"Total time: {time_end - time_start:.2f} seconds")
